In [1]:
# Cell 1: Verify GPU availability
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU")

PyTorch version : 2.10.0+cpu
CUDA available  : False
⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU


In [1]:
# Paste this as Cell 1 BEFORE installing anything
import torch
print(torch.__version__)          # will show old version — that's fine
print(torch.cuda.is_available())  # should now say True

2.10.0+cu128
True


In [2]:
# Cell 2: Install pinned dependencies for Mini GPT project
# Runtime: T4 GPU confirmed — safe to install CUDA-enabled wheels

import subprocess, sys

def install(package):
    print(f"  Installing {package}...", end=" ", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", package,
         "--quiet", "--no-warn-script-location"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("done")
    else:
        print(f"FAILED\n{result.stderr[-300:]}")

packages = [
    "torch==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118",
    "torchvision==0.16.0+cu118 --index-url https://download.pytorch.org/whl/cu118",
    "torchaudio==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118",
    "transformers==4.36.0",
    "datasets==2.16.1",
    "gradio==4.0.0",
    "numpy==1.24.4",
]

print("Installing packages (this takes 2-4 mins)...\n")
for pkg in packages:
    install(pkg)

print("\nDone! Now go to: Runtime → Restart runtime")
print("Then run Cell 3 to verify versions.")

Installing packages (this takes 2-4 mins)...

  Installing torch==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118... FAILED
oad.pytorch.org/whl/cu118': Expected end or semicolon (after version specifier)
    torch==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118
         ~~~~~~~~~~~~~~^
Hint: It looks like a path. File 'torch==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118' does not exist.

  Installing torchvision==0.16.0+cu118 --index-url https://download.pytorch.org/whl/cu118... FAILED
u118': Expected end or semicolon (after version specifier)
    torchvision==0.16.0+cu118 --index-url https://download.pytorch.org/whl/cu118
               ~~~~~~~~~~~~~~~^
Hint: It looks like a path. File 'torchvision==0.16.0+cu118 --index-url https://download.pytorch.org/whl/cu118' does not exist.

  Installing torchaudio==2.1.0+cu118 --index-url https://download.pytorch.org/whl/cu118... FAILED
/whl/cu118': Expected end or semicolon (after version specifi

In [2]:
# Cell 3 (revised): Accept Colab's native versions — all are compatible

import torch, torchvision, torchaudio
import transformers, datasets, gradio, numpy

print("Package versions in use:\n")
libs = {
    "torch"        : torch.__version__,
    "torchvision"  : torchvision.__version__,
    "torchaudio"   : torchaudio.__version__,
    "transformers" : transformers.__version__,
    "datasets"     : datasets.__version__,
    "gradio"       : gradio.__version__,
    "numpy"        : numpy.__version__,
}
for lib, ver in libs.items():
    print(f"  {lib:<14} {ver}")

# Compatibility gate — these are the minimums we actually need
import sys
assert torch.__version__ >= "2.1", "Need torch >= 2.1"
assert transformers.__version__ == "4.36.0", "Need transformers 4.36.0"
assert datasets.__version__  == "2.16.1",  "Need datasets 2.16.1"
assert gradio.__version__    == "4.0.0",   "Need gradio 4.0.0"

# Device setup — this variable is reused in ALL future cells
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nDevice   : {str(DEVICE).upper()}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA     : {torch.version.cuda}")
print(f"\nAll compatibility checks passed!")
print("Step 1 complete — ready for Step 2: Dataset Loading")

Package versions in use:

  torch          2.10.0+cu128
  torchvision    0.25.0+cu128
  torchaudio     2.10.0+cu128
  transformers   4.36.0
  datasets       2.16.1
  gradio         4.0.0
  numpy          1.26.4

Device   : CUDA
GPU      : Tesla T4
VRAM     : 15.6 GB
CUDA     : 12.8

All compatibility checks passed!
Step 1 complete — ready for Step 2: Dataset Loading


In [3]:
# Cell 4: Load TinyStories dataset (small subset ~10MB)
# Uses HuggingFace datasets library — no manual download needed

from datasets import load_dataset
import random, os

print("Loading TinyStories from HuggingFace Hub...")
print("(downloads ~1-2 GB full dataset, then we slice it)\n")

# Load only the train split — we don't need validation for this project
dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    trust_remote_code=True
)

print(f"Full dataset size : {len(dataset):,} stories")
print(f"Features          : {dataset.features}")
print(f"\nSample story:\n{'-'*50}")
print(dataset[0]['text'][:300], "...")
print('-'*50)

# ── Slice a manageable subset ──────────────────────────
SUBSET_SIZE = 10_000   # ~10MB of text — fast to train, enough to learn patterns

random.seed(42)
indices = random.sample(range(len(dataset)), SUBSET_SIZE)
subset  = dataset.select(indices)

print(f"\nSubset size : {SUBSET_SIZE:,} stories")

# Compute approximate text size
total_chars = sum(len(ex['text']) for ex in subset)
print(f"Total chars : {total_chars:,}  (~{total_chars/1e6:.1f} MB of text)")
print(f"\nStep 2 complete — raw data loaded. Proceeding to cleaning...")

Loading TinyStories from HuggingFace Hub...
(downloads ~1-2 GB full dataset, then we slice it)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Full dataset size : 2,119,719 stories
Features          : {'text': Value(dtype='string', id=None)}

Sample story:
--------------------------------------------------
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and  ...
--------------------------------------------------

Subset size : 10,000 stories
Total chars : 8,970,132  (~9.0 MB of text)

Step 2 complete — raw data loaded. Proceeding to cleaning...


In [4]:
# Cell 5: Data cleaning — character-level models are sensitive to noise
# Every unique character = a vocab entry, so we want a clean, minimal charset

import re
from collections import Counter

# ── 1. Extract raw texts ───────────────────────────────
raw_texts = [ex['text'] for ex in subset]

def analyze_charset(texts, label):
    all_text = ''.join(texts)
    char_counts = Counter(all_text)
    total_chars = len(all_text)
    unique_chars = len(char_counts)
    # Find non-ASCII or suspicious characters
    weird = {c: n for c, n in char_counts.items()
             if ord(c) > 127 or c in '\x00\x01\x02\x03\x0b\x0c\x0e\x0f'}
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  Total characters : {total_chars:>10,}")
    print(f"  Unique characters: {unique_chars:>10,}")
    print(f"  Vocab size       : {unique_chars:>10,}")
    print(f"  Non-ASCII / weird: {len(weird):>10,} types")
    if weird:
        print(f"  Weird chars      : ", end="")
        for c, n in list(weird.items())[:10]:
            print(f"U+{ord(c):04X}({n})", end=" ")
        print()
    # Show top-30 most common chars
    print(f"\n  Top 20 most frequent characters:")
    for char, count in char_counts.most_common(20):
        bar = '█' * int(40 * count / char_counts.most_common(1)[0][1])
        display = repr(char) if char in (' ', '\n', '\t') else char
        print(f"    {display:<6} {count:>8,}  {bar}")
    return all_text, char_counts

# ── BEFORE cleaning ────────────────────────────────────
raw_all, raw_counts = analyze_charset(raw_texts, "BEFORE CLEANING")

# ── 2. Cleaning pipeline ───────────────────────────────
def clean_text(text):
    # Step A: Normalize unicode punctuation → ASCII equivalents
    replacements = {
        '\u2018': "'", '\u2019': "'",   # curly single quotes
        '\u201c': '"', '\u201d': '"',   # curly double quotes
        '\u2013': '-', '\u2014': '-',   # en-dash, em-dash
        '\u2026': '...',                # ellipsis character
        '\u00e9': 'e', '\u00e8': 'e',  # accented e
        '\u00e0': 'a', '\u00e1': 'a',  # accented a
        '\u00f3': 'o', '\u00f6': 'o',  # accented o
        '\u00fa': 'u', '\u00fc': 'u',  # accented u
        '\u00ed': 'i', '\u00ef': 'i',  # accented i
        '\u00f1': 'n',                  # n-tilde
        '\u00e7': 'c',                  # c-cedilla
        '\xa0':   ' ',                  # non-breaking space
    }
    for orig, repl in replacements.items():
        text = text.replace(orig, repl)

    # Step B: Remove non-printable / control characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # Step C: Collapse 3+ newlines → 2 (preserve paragraph breaks)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Step D: Collapse multiple spaces → single space
    text = re.sub(r'[ \t]{2,}', ' ', text)

    # Step E: Strip any remaining non-ASCII (anything above 127)
    text = re.sub(r'[^\x00-\x7f]', '', text)

    # Step F: Strip leading/trailing whitespace per story
    text = text.strip()

    return text

# Apply cleaning
cleaned_texts = [clean_text(t) for t in raw_texts]

# Remove any stories that became too short after cleaning
min_length = 50
cleaned_texts = [t for t in cleaned_texts if len(t) >= min_length]

# ── AFTER cleaning ─────────────────────────────────────
clean_all, clean_counts = analyze_charset(cleaned_texts, "AFTER CLEANING")

# ── 3. Summary diff ────────────────────────────────────
print(f"\n{'='*50}")
print(f"  CLEANING SUMMARY")
print(f"{'='*50}")
print(f"  Stories  : {len(raw_texts):,}  →  {len(cleaned_texts):,}  "
      f"(removed {len(raw_texts)-len(cleaned_texts)})")
print(f"  Chars    : {len(raw_all):,}  →  {len(clean_all):,}  "
      f"(removed {len(raw_all)-len(clean_all):,})")
print(f"  Vocab    : {len(raw_counts):,}  →  {len(clean_counts):,}  "
      f"(reduced by {len(raw_counts)-len(clean_counts)})")
print(f"\n  Final clean charset ({len(clean_counts)} chars):")
final_chars = sorted(clean_counts.keys())
print(f"  {''.join(final_chars)}")
print(f"\nStep 3 complete — data is clean. Ready for tokenizer!")


  BEFORE CLEANING
  Total characters :  8,970,132
  Unique characters:         93
  Vocab size       :         93
  Non-ASCII / weird:         16 types
  Weird chars      : U+00E2(3451) U+20AC(3451) U+0153(1236) U+2122(878) U+201D(17) U+00C3(2) U+00A9(2) U+00EF(1) U+00BB(1) U+00BF(1) 

  Top 20 most frequent characters:
    ' '    1,710,227  ████████████████████████████████████████
    e       854,257  ███████████████████
    a       604,968  ██████████████
    t       559,718  █████████████
    o       468,688  ██████████
    h       440,746  ██████████
    n       416,611  █████████
    i       384,848  █████████
    d       383,522  ████████
    s       372,106  ████████
    r       326,646  ███████
    l       275,919  ██████
    y       205,956  ████
    m       178,459  ████
    w       175,394  ████
    .       172,572  ████
    u       169,608  ███
    g       130,586  ███
    c       129,151  ███
    p       125,772  ██

  AFTER CLEANING
  Total characters :  8,961,007
  Uniq

In [5]:
# Cell 6: Character-level tokenizer
# Maps every unique character ↔ integer index
# This is the entire "tokenizer" for a char-level model — no BPE, no subwords

# ── 1. Build vocabulary from cleaned data ──────────────
all_text = '\n'.join(cleaned_texts)   # join all stories with newline separator

# Sorted for determinism — same vocab every run
vocab = sorted(set(all_text))
vocab_size = len(vocab)

print(f"Vocabulary size : {vocab_size}")
print(f"Vocabulary      : {repr(''.join(vocab))}")

# ── 2. Build lookup tables ─────────────────────────────
stoi = {ch: i for i, ch in enumerate(vocab)}   # char → index
itos = {i: ch for i, ch in enumerate(vocab)}   # index → char

print(f"\nSample mappings (char → index):")
for ch in [' ', 'a', 'e', '.', '!', '?', '\n', 'A']:
    print(f"  {repr(ch):<6} → {stoi[ch]}")

# ── 3. Encode / decode functions ───────────────────────
def encode(text):
    """Convert string → list of integers."""
    return [stoi[ch] for ch in text]

def decode(indices):
    """Convert list of integers → string."""
    return ''.join(itos[i] for i in indices)

# ── 4. Smoke test ──────────────────────────────────────
test_str = "Once upon a time!"
encoded  = encode(test_str)
decoded  = decode(encoded)

print(f"\nTokenizer smoke test:")
print(f"  Input   : {repr(test_str)}")
print(f"  Encoded : {encoded}")
print(f"  Decoded : {repr(decoded)}")
print(f"  Match   : {test_str == decoded}")

# ── 5. Encode entire dataset into one long tensor ──────
import torch

print(f"\nEncoding full dataset to tensor...")
full_encoded = encode(all_text)
data = torch.tensor(full_encoded, dtype=torch.long)

print(f"  Dataset tensor shape : {data.shape}")
print(f"  dtype                : {data.dtype}")
print(f"  Min / Max token id   : {data.min().item()} / {data.max().item()}")
print(f"  Expected max         : {vocab_size - 1}")

# ── 6. Train / validation split ────────────────────────
split_idx  = int(0.9 * len(data))
train_data = data[:split_idx]
val_data   = data[split_idx:]

print(f"\nData split:")
print(f"  Train tokens : {len(train_data):>10,}  ({100*len(train_data)/len(data):.0f}%)")
print(f"  Val tokens   : {len(val_data):>10,}  ({100*len(val_data)/len(data):.0f}%)")

# ── 7. Verify a round-trip on real data ────────────────
sample_ids   = train_data[:200].tolist()
sample_text  = decode(sample_ids)
print(f"\nFirst 200 tokens decoded from train set:")
print(f"{'─'*50}")
print(sample_text)
print(f"{'─'*50}")

print(f"\nStep 4 complete — tokenizer built and data encoded!")
print(f"vocab_size={vocab_size} is saved — needed for model definition.")

Vocabulary size : 77
Vocabulary      : '\n !"$\'+,-./012345689:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ_`abcdefghijklmnopqrstuvwxyz'

Sample mappings (char → index):
  ' '    → 1
  'a'    → 51
  'e'    → 55
  '.'    → 9
  '!'    → 2
  '?'    → 22
  '\n'   → 0
  'A'    → 23

Tokenizer smoke test:
  Input   : 'Once upon a time!'
  Encoded : [37, 64, 53, 55, 1, 71, 66, 65, 64, 1, 51, 1, 70, 59, 63, 55, 2]
  Decoded : 'Once upon a time!'
  Match   : True

Encoding full dataset to tensor...
  Dataset tensor shape : torch.Size([8971002])
  dtype                : torch.int64
  Min / Max token id   : 0 / 76
  Expected max         : 76

Data split:
  Train tokens :  8,073,901  (90%)
  Val tokens   :    897,101  (10%)

First 200 tokens decoded from train set:
──────────────────────────────────────────────────
One day, a little boy named Tim was very excited. He saw a big gray dog in the park. The dog was playing with a ball. Tim wanted to play with the dog too.

Tim went to his mom and asked her to explain


In [6]:
# Cell 7: Mini GPT — full decoder-only transformer from scratch
# Architecture: 4 layers, 4 heads, embed_dim=128, context=128 chars

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ── Hyperparameters ────────────────────────────────────
BLOCK_SIZE  = 128    # context window (characters)
EMBED_DIM   = 128    # embedding dimension
N_HEADS     = 4      # attention heads (head_dim = 128/4 = 32)
N_LAYERS    = 4      # number of decoder blocks
FFN_DIM     = 512    # feed-forward hidden dim (4× embed)
DROPOUT     = 0.1    # regularization dropout

# Already defined in Cell 6 — confirmed: 77
print(f"Building model with vocab_size={vocab_size}")

# ══════════════════════════════════════════════════════
# 1. CAUSAL SELF-ATTENTION
# ══════════════════════════════════════════════════════
class CausalSelfAttention(nn.Module):
    """
    Multi-head self-attention with causal (autoregressive) mask.
    Each token can only attend to itself and tokens BEFORE it —
    never to future tokens. This is enforced by the triangular mask.
    """
    def __init__(self, embed_dim, n_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"

        self.n_heads  = n_heads
        self.head_dim = embed_dim // n_heads   # 128 // 4 = 32

        # Single matrix for Q, K, V projections (3× for efficiency)
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj  = nn.Linear(embed_dim, embed_dim,     bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop= nn.Dropout(dropout)

        # Causal mask: lower-triangular matrix of ones
        # register_buffer = saved with model but not a trainable parameter
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
            .view(1, 1, BLOCK_SIZE, BLOCK_SIZE)   # shape: (1,1,T,T)
        )

    def forward(self, x):
        B, T, C = x.shape   # batch, sequence_len, embed_dim

        # Project to Q, K, V and split into heads
        qkv = self.qkv_proj(x)                          # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)                   # each: (B, T, C)

        # Reshape to (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        scale  = 1.0 / math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) * scale      # (B, H, T, T)

        # Apply causal mask — future positions get -inf → softmax → 0
        scores = scores.masked_fill(
            self.mask[:, :, :T, :T] == 0, float('-inf')
        )
        weights = F.softmax(scores, dim=-1)              # (B, H, T, T)
        weights = self.attn_drop(weights)

        # Weighted sum of values
        out = weights @ v                                # (B, H, T, head_dim)
        out = out.transpose(1, 2).contiguous()           # (B, T, H, head_dim)
        out = out.view(B, T, C)                          # (B, T, C)  re-merge heads
        return self.resid_drop(self.out_proj(out))

# ══════════════════════════════════════════════════════
# 2. FEED-FORWARD NETWORK
# ══════════════════════════════════════════════════════
class FeedForward(nn.Module):
    """
    Position-wise FFN: two linear layers with GELU activation.
    Expands embed_dim → 4× → embed_dim.
    This is where the model stores factual/pattern knowledge.
    """
    def __init__(self, embed_dim, ffn_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# ══════════════════════════════════════════════════════
# 3. DECODER BLOCK
# ══════════════════════════════════════════════════════
class DecoderBlock(nn.Module):
    """
    One full transformer decoder block:
      x = x + Attention(LayerNorm(x))   ← pre-norm residual
      x = x + FFN(LayerNorm(x))         ← pre-norm residual
    Pre-norm (norm before sublayer) trains more stably than post-norm.
    """
    def __init__(self, embed_dim, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, n_heads, dropout)
        self.ln2  = nn.LayerNorm(embed_dim)
        self.ffn  = FeedForward(embed_dim, ffn_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # attention sub-layer + residual
        x = x + self.ffn(self.ln2(x))    # FFN sub-layer + residual
        return x

# ══════════════════════════════════════════════════════
# 4. MINI GPT
# ══════════════════════════════════════════════════════
class MiniGPT(nn.Module):
    """
    Full decoder-only transformer language model.
    Input  : token indices (B, T)
    Output : logits over vocabulary (B, T, vocab_size)
    """
    def __init__(self, vocab_size, embed_dim, n_heads,
                 n_layers, ffn_dim, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb   = nn.Embedding(block_size, embed_dim)
        self.drop      = nn.Dropout(dropout)

        self.blocks = nn.Sequential(
            *[DecoderBlock(embed_dim, n_heads, ffn_dim, dropout)
              for _ in range(n_layers)]
        )

        self.ln_final = nn.LayerNorm(embed_dim)
        self.lm_head  = nn.Linear(embed_dim, vocab_size, bias=False)

        # Weight tying: share token embedding and lm_head weights
        # This is standard GPT practice — reduces parameters, improves quality
        self.lm_head.weight = self.token_emb.weight

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, \
            f"Sequence length {T} exceeds block_size {self.block_size}"

        # Token + positional embeddings
        positions = torch.arange(T, device=idx.device)   # (T,)
        x = self.drop(
            self.token_emb(idx) + self.pos_emb(positions)
        )                                                  # (B, T, C)

        # Pass through all decoder blocks
        x = self.blocks(x)                                # (B, T, C)
        x = self.ln_final(x)                              # (B, T, C)
        logits = self.lm_head(x)                          # (B, T, vocab_size)

        # Compute loss if targets provided (training mode)
        loss = None
        if targets is not None:
            # Flatten for cross-entropy: (B*T, vocab_size) vs (B*T,)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

# ── Instantiate and inspect ────────────────────────────
model = MiniGPT(
    vocab_size = vocab_size,
    embed_dim  = EMBED_DIM,
    n_heads    = N_HEADS,
    n_layers   = N_LAYERS,
    ffn_dim    = FFN_DIM,
    block_size = BLOCK_SIZE,
    dropout    = DROPOUT,
).to(DEVICE)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel architecture:\n{model}\n")
print(f"{'─'*40}")
print(f"Total parameters     : {total_params:>10,}")
print(f"Trainable parameters : {trainable_params:>10,}")
print(f"Model size (approx)  : {total_params * 4 / 1e6:.2f} MB  (float32)")
print(f"Device               : {next(model.parameters()).device}")

# ── Sanity-check forward pass ──────────────────────────
print(f"\nRunning forward pass sanity check...")
dummy_input  = torch.randint(0, vocab_size, (2, BLOCK_SIZE)).to(DEVICE)
dummy_target = torch.randint(0, vocab_size, (2, BLOCK_SIZE)).to(DEVICE)
with torch.no_grad():
    logits, loss = model(dummy_input, dummy_target)

print(f"  Input shape  : {dummy_input.shape}")
print(f"  Logits shape : {logits.shape}   (B, T, vocab_size)")
print(f"  Loss         : {loss.item():.4f}  "
      f"(random baseline ≈ {math.log(vocab_size):.4f})")
print(f"\nStep 5 complete — Mini GPT built and verified!")
print(f"Loss ≈ log(vocab_size) confirms random init is correct.")

Building model with vocab_size=77

Model architecture:
MiniGPT(
  (token_emb): Embedding(77, 128)
  (pos_emb): Embedding(128, 128)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (qkv_proj): Linear(in_features=128, out_features=384, bias=False)
        (out_proj): Linear(in_features=128, out_features=128, bias=False)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (1): DecoderBlock(
      (ln1): LayerNorm((128,), ep

In [7]:
# Cell 8: Training loop — AdamW optimizer, cross-entropy loss
# Estimated time: ~15-20 mins on T4 GPU for 5000 steps

import time
import math

# ── Training hyperparameters ───────────────────────────
BATCH_SIZE   = 32       # sequences per step
BLOCK_SIZE   = 128      # already defined — context window
MAX_STEPS    = 5000     # total training steps
EVAL_EVERY   = 500      # evaluate on val set every N steps
EVAL_STEPS   = 50       # how many val batches to average
LR           = 3e-4     # learning rate (AdamW default for small transformers)
WARMUP_STEPS = 200      # linear LR warmup to prevent early instability
MIN_LR       = 3e-5     # cosine decay floor (10% of max LR)

# ── Data batch sampler ─────────────────────────────────
def get_batch(split):
    """
    Sample a random batch of (input, target) pairs.
    Input  : tokens[i : i+BLOCK_SIZE]
    Target : tokens[i+1 : i+BLOCK_SIZE+1]  ← shifted by 1
    This is the language modelling objective: predict next character.
    """
    data = train_data if split == 'train' else val_data
    # Random starting positions — one per batch item
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x  = torch.stack([data[i     : i + BLOCK_SIZE    ] for i in ix])
    y  = torch.stack([data[i + 1 : i + BLOCK_SIZE + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# ── LR schedule: linear warmup + cosine decay ──────────
def get_lr(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS          # linear warmup
    # Cosine decay from LR → MIN_LR
    progress = (step - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    coeff    = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + coeff * (LR - MIN_LR)

# ── Validation loss estimator ──────────────────────────
@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        split_losses = []
        for _ in range(EVAL_STEPS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            split_losses.append(loss.item())
        losses[split] = sum(split_losses) / len(split_losses)
    model.train()
    return losses

# ── Optimizer ──────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = LR,
    betas        = (0.9, 0.95),   # standard for transformers
    weight_decay = 0.1,           # L2 regularization on weights
    eps          = 1e-8,
)

# ── Training loop ──────────────────────────────────────
print(f"Training Mini GPT")
print(f"{'─'*55}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Block size   : {BLOCK_SIZE}")
print(f"  Max steps    : {MAX_STEPS:,}")
print(f"  Eval every   : {EVAL_EVERY} steps")
print(f"  Learning rate: {LR} (warmup {WARMUP_STEPS} steps, cosine decay)")
print(f"{'─'*55}\n")

train_losses = []
val_losses   = []
log_steps    = []

model.train()
t0 = time.time()

for step in range(MAX_STEPS):

    # ── LR update ──────────────────────────────────────
    lr_now = get_lr(step)
    for g in optimizer.param_groups:
        g['lr'] = lr_now

    # ── Evaluate periodically ──────────────────────────
    if step % EVAL_EVERY == 0 or step == MAX_STEPS - 1:
        losses   = estimate_loss()
        elapsed  = time.time() - t0
        steps_ps = (step + 1) / elapsed if elapsed > 0 else 0
        eta_s    = (MAX_STEPS - step) / steps_ps if steps_ps > 0 else 0

        train_losses.append(losses['train'])
        val_losses.append(losses['val'])
        log_steps.append(step)

        print(f"  step {step:>5} | "
              f"train {losses['train']:.4f} | "
              f"val {losses['val']:.4f} | "
              f"lr {lr_now:.2e} | "
              f"ETA {eta_s/60:.1f} min")

    # ── Forward + backward ─────────────────────────────
    x, y       = get_batch('train')
    logits, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()
    loss.backward()

    # Gradient clipping — prevents exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

total_time = time.time() - t0
print(f"\n{'─'*55}")
print(f"Training complete in {total_time/60:.1f} minutes")
print(f"Final train loss : {train_losses[-1]:.4f}")
print(f"Final val loss   : {val_losses[-1]:.4f}")
print(f"\nStep 6 complete — model trained!")
print(f"Proceeding to text generation...")

Training Mini GPT
───────────────────────────────────────────────────────
  Batch size   : 32
  Block size   : 128
  Max steps    : 5,000
  Eval every   : 500 steps
  Learning rate: 0.0003 (warmup 200 steps, cosine decay)
───────────────────────────────────────────────────────

  step     0 | train 4.3846 | val 4.3849 | lr 1.50e-06 | ETA 74.8 min
  step   500 | train 2.1652 | val 2.1725 | lr 2.97e-04 | ETA 1.5 min
  step  1000 | train 1.7949 | val 1.8025 | lr 2.82e-04 | ETA 1.2 min
  step  1500 | train 1.5670 | val 1.5720 | lr 2.54e-04 | ETA 1.0 min
  step  2000 | train 1.4603 | val 1.4517 | lr 2.17e-04 | ETA 0.9 min
  step  2500 | train 1.3896 | val 1.3818 | lr 1.74e-04 | ETA 0.7 min
  step  3000 | train 1.3113 | val 1.3257 | lr 1.30e-04 | ETA 0.6 min
  step  3500 | train 1.2868 | val 1.2858 | lr 9.00e-05 | ETA 0.4 min
  step  4000 | train 1.2590 | val 1.2598 | lr 5.79e-05 | ETA 0.3 min
  step  4500 | train 1.2452 | val 1.2592 | lr 3.72e-05 | ETA 0.1 min
  step  4999 | train 1.2386 | 

In [8]:
# Cell 9: Autoregressive text generation with temperature sampling
# The model generates one character at a time, feeding each output
# back as the next input — this is exactly how GPT works.

import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(
    prompt      : str,
    max_new_chars: int  = 300,
    temperature : float = 1.0,
    top_k       : int   = None,
):
    """
    Generate text autoregressively from a prompt.

    temperature: controls randomness
        < 1.0  → sharper distribution, more predictable/repetitive
        = 1.0  → raw model distribution
        > 1.0  → flatter distribution, more creative/chaotic

    top_k: if set, only sample from the top-k most likely tokens
        Helps prevent very unlikely characters from being sampled.
    """
    model.eval()

    # Encode prompt → token indices
    input_ids = encode(prompt)
    idx = torch.tensor(input_ids, dtype=torch.long,
                       device=DEVICE).unsqueeze(0)   # shape: (1, T)

    generated_chars = []

    for _ in range(max_new_chars):
        # Crop to block_size if sequence gets too long
        idx_cond = idx[:, -BLOCK_SIZE:]              # (1, T) — max 128 tokens

        # Forward pass — get logits for all positions
        logits, _ = model(idx_cond)                  # (1, T, vocab_size)

        # Take logits at the LAST position only (next character prediction)
        logits = logits[:, -1, :]                    # (1, vocab_size)

        # Apply temperature
        logits = logits / temperature

        # Optional top-k filtering
        if top_k is not None:
            # Zero out all logits except the top-k
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < values[:, [-1]]] = float('-inf')

        # Convert logits → probabilities
        probs = F.softmax(logits, dim=-1)            # (1, vocab_size)

        # Sample next token from distribution
        next_id = torch.multinomial(probs, num_samples=1)  # (1, 1)

        # Append to sequence
        idx = torch.cat([idx, next_id], dim=1)       # (1, T+1)
        generated_chars.append(itos[next_id.item()])

    return prompt + ''.join(generated_chars)


# ── Test generation at multiple temperatures ───────────
prompts = [
    "Once upon a time",
    "The little girl",
    "Tom was sad because",
]

temperatures = [0.7, 1.0, 1.2]

print("=" * 60)
print("  TEXT GENERATION SAMPLES")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: \"{prompt}\"")
    print("─" * 60)
    for temp in temperatures:
        result = generate(
            prompt       = prompt,
            max_new_chars= 200,
            temperature  = temp,
            top_k        = 40,
        )
        print(f"\n  [temp={temp}]")
        print(f"  {result}")
    print()

# ── Show effect of temperature more clearly ────────────
print("=" * 60)
print("  TEMPERATURE EFFECT (same prompt, same model)")
print("=" * 60)
demo_prompt = "One day, a little boy"
print(f"\nPrompt: \"{demo_prompt}\"\n")

for temp in [0.5, 0.8, 1.0, 1.3]:
    out = generate(demo_prompt, max_new_chars=120,
                   temperature=temp, top_k=40)
    print(f"  temp={temp}:")
    print(f"  {out}\n")

print("Step 7 complete — text generation working!")
print("Ready for Step 8: Gradio UI")

  TEXT GENERATION SAMPLES

Prompt: "Once upon a time"
────────────────────────────────────────────────────────────

  [temp=0.7]
  Once upon a time, there was a little girl was a girl named Sam were happy. She put it was very supped so her because drove to si to be not his for a niside to play with his looking not on it. She liked to her mom and

  [temp=1.0]
  Once upon a time, there we was a minde. They started to like to tay. They are a big made blot be nice it. They calleg a cloud filly in their find. Lily said, she sturrous shiny. He says nod.

"You says, "Would not so

  [temp=1.2]
  Once upon a time, there went to the Maxa was pied in the forget. One day, she named Timmy, thebat bird he ould too. Sue he had a feen soly.

When rombed he huged and sties arround. Glebs, it is snuckly thought inside


Prompt: "The little girl"
────────────────────────────────────────────────────────────

  [temp=0.7]
  The little girl was so happy.

They noded the make a saw she was friends. She rea